# Extracción y Diagnóstico de Datos de Establecimientos del MINEDUC

Este notebook automatiza la descarga de registros de establecimientos educativos desde el portal del MINEDUC de Guatemala utilizando Selenium, limpia la información y ejecuta un diagnóstico paso a paso de calidad de datos.

## 1. Importación de Librerías y Configuración Global

In [1]:
import os
import time
import glob
import re
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.action_chains import ActionChains

# Parámetros globales de búsqueda y almacenamiento
BASE_URL = "https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/"
CARPETA_DESCARGAS = os.path.join(os.getcwd(), "descargas")
CSV_SALIDA = "mineduc_establecimientos_unificado.csv"

DEPARTAMENTOS = [
    "ALTA VERAPAZ", "BAJA VERAPAZ", "CHIMALTENANGO", "CHIQUIMULA",
    "CIUDAD CAPITAL", "EL PROGRESO", "ESCUINTLA", "GUATEMALA",
    "HUEHUETENANGO", "IZABAL", "JALAPA", "JUTIAPA", "PETEN",
    "QUETZALTENANGO", "QUICHE", "RETALHULEU", "SACATEPEQUEZ",
    "SAN MARCOS", "SANTA ROSA", "SOLOLA", "SUCHITEPEQUEZ",
    "TOTONICAPAN", "ZACAPA"
]

NIVELES = ["BASICO", "DIVERSIFICADO"]

## 2. Funciones Auxiliares para Web Scraping

In [2]:
def encontrar_select_por_nombre(driver, nombre_parcial):
    selects = driver.find_elements(By.TAG_NAME, "select")
    for sel in selects:
        name = sel.get_attribute("name")
        if name and nombre_parcial in name:
            return sel
    return None

def encontrar_boton_por_src(driver, src_parcial):
    inputs = driver.find_elements(By.XPATH, "//input[@type='image']")
    for inp in inputs:
        src = inp.get_attribute("src")
        if src and src_parcial in src:
            return inp
    return None

def hacer_scroll_hasta_elemento(driver, elemento):
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", elemento)
    time.sleep(0.5)

def clic_seguro(driver, elemento):
    try:
        elemento.click()
    except Exception:
        ActionChains(driver).move_to_element(elemento).click().perform()

def esperar_descarga(carpeta, timeout=30):
    archivos_iniciales = set(glob.glob(os.path.join(carpeta, "*.xls")))
    start = time.time()
    while time.time() - start < timeout:
        archivos_actuales = set(glob.glob(os.path.join(carpeta, "*.xls")))
        nuevos = archivos_actuales - archivos_iniciales
        if nuevos:
            return max(nuevos, key=os.path.getctime)
        time.sleep(1)
    raise TimeoutError("No se detectó la descarga del archivo .xls")

def leer_tabla_desde_archivo_xls(ruta_archivo):
    with open(ruta_archivo, 'r', encoding='latin1') as f:
        contenido = f.read()
    tablas = pd.read_html(contenido)
    if not tablas:
        raise ValueError("No se encontraron tablas en el archivo")
    df = max(tablas, key=lambda t: t.shape[0])
    df.columns = df.iloc[0]
    df = df[1:].reset_index(drop=True)
    df = df.dropna(how='all')
    return df

## 3. Extracción y Carga Inicial de Datos

In [3]:
def descargar_departamento_nivel(driver, departamento, nivel):
    driver.get(BASE_URL)
    wait = WebDriverWait(driver, 15)

    select_depto = encontrar_select_por_nombre(driver, "cmbDepartamento")
    if not select_depto:
        raise Exception("No se encontró el select de Departamento")
    Select(select_depto).select_by_visible_text(departamento)
    time.sleep(1.5)

    select_nivel = encontrar_select_por_nombre(driver, "cmbNivel")
    if not select_nivel:
        raise Exception("No se encontró el select de Nivel")
    Select(select_nivel).select_by_visible_text(nivel)
    time.sleep(0.5)

    boton_buscar = driver.find_element(By.XPATH, "//input[@type='image' and contains(@src, 'seleccionar_estab.gif')]")
    hacer_scroll_hasta_elemento(driver, boton_buscar)
    clic_seguro(driver, boton_buscar)
    time.sleep(4)

    try:
        wait.until(EC.presence_of_element_located((By.XPATH, "//table")))
    except TimeoutException:
        print(f"  Sin resultados para {departamento} - {nivel}")
        return None

    boton_excel = encontrar_boton_por_src(driver, "export_excel.gif")
    if not boton_excel:
        try:
            boton_excel = driver.find_element(By.XPATH, "//*[contains(text(),'Generar Archivo de Excel')]")
        except Exception:
            pass
    if not boton_excel:
        raise Exception("No se encontró el botón de Excel")

    hacer_scroll_hasta_elemento(driver, boton_excel)
    wait.until(EC.element_to_be_clickable(boton_excel))
    boton_excel.click()

    ruta_archivo = esperar_descarga(CARPETA_DESCARGAS, timeout=30)
    nuevo_nombre = f"{departamento}_{nivel}.xls"
    nueva_ruta = os.path.join(CARPETA_DESCARGAS, nuevo_nombre)
    if os.path.exists(nueva_ruta):
        base, ext = os.path.splitext(nuevo_nombre)
        nueva_ruta = os.path.join(CARPETA_DESCARGAS, f"{base}_{int(time.time())}{ext}")
    os.rename(ruta_archivo, nueva_ruta)
    print(f"  Descargado: {os.path.basename(nueva_ruta)}")
    return nueva_ruta

def descargar_y_unificar():
    os.makedirs(CARPETA_DESCARGAS, exist_ok=True)

    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    prefs = {
        "download.default_directory": CARPETA_DESCARGAS,
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    }
    options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(options=options)
    archivos_descargados = []

    try:
        print("Iniciando descarga de todos los departamentos (Básico y Diversificado)...")
        for depto in DEPARTAMENTOS:
            for nivel in NIVELES:
                try:
                    print(f"Procesando {depto} - {nivel}...")
                    ruta = descargar_departamento_nivel(driver, depto, nivel)
                    if ruta:
                        archivos_descargados.append(ruta)
                except Exception as e:
                    print(f"  ERROR: {e}")
                time.sleep(2)

        if not archivos_descargados:
            raise SystemExit("No se descargó ningún archivo.")

        dataframes = []
        for archivo in archivos_descargados:
            try:
                df = leer_tabla_desde_archivo_xls(archivo)
                nombre_base = os.path.basename(archivo).replace(".xls", "")
                partes = nombre_base.split("_")
                if len(partes) >= 2:
                    df["DEPARTAMENTO_CONSULTADO"] = partes[0]
                    df["NIVEL_CONSULTADO"] = partes[1]
                dataframes.append(df)
            except Exception as e:
                print(f"Error al leer {archivo}: {e}")

        df_unificado = pd.concat(dataframes, ignore_index=True)
        df_unificado.columns = [str(c).strip().upper() for c in df_unificado.columns]

        if "CODIGO" in df_unificado.columns:
            df_unificado = df_unificado[df_unificado["CODIGO"].notna()]
            df_unificado = df_unificado[df_unificado["CODIGO"].astype(str).str.strip() != ""]

        categoricas = ["DEPARTAMENTO", "MUNICIPIO", "NIVEL", "SECTOR", "AREA",
                       "STATUS", "MODALIDAD", "JORNADA", "PLAN", "DEPARTAMENTAL"]
        for col in categoricas:
            if col in df_unificado.columns:
                df_unificado[col] = df_unificado[col].astype(str).str.upper().str.strip()
                df_unificado[col] = df_unificado[col].replace({"NAN": pd.NA, "NONE": pd.NA, "---": pd.NA})

        if "TELEFONO" in df_unificado.columns:
            df_unificado["TELEFONO"] = df_unificado["TELEFONO"].astype(str).str.replace(r"\.0$", "", regex=True)
            df_unificado["TELEFONO"] = df_unificado["TELEFONO"].replace({"nan": pd.NA, "<NA>": pd.NA})

        if "CODIGO" in df_unificado.columns:
            df_unificado["_no_nulos"] = df_unificado.notna().sum(axis=1)
            df_unificado = df_unificado.sort_values("_no_nulos", ascending=False).drop_duplicates(subset=["CODIGO"]).drop(columns="_no_nulos")

        df_unificado.to_csv(CSV_SALIDA, index=False, encoding="utf-8-sig")
        return df_unificado
    finally:
        driver.quit()

# Obtención del Dataset
if os.path.exists(CSV_SALIDA):
    print(f"Cargando archivo existente '{CSV_SALIDA}'...")
    df = pd.read_csv(CSV_SALIDA, encoding='utf-8-sig')
else:
    print("Descargando datos...")
    df = descargar_y_unificar()

Cargando archivo existente 'mineduc_establecimientos_unificado.csv'...


## 4. Diagnóstico de Calidad de Datos

### a. Número de registros y variables
**Descripción:** Evalúa la dimensión total del conjunto de datos identificando la cantidad total de filas (registros) y columnas (variables).

In [4]:
print("a. Número de registros y variables:")
print(f"   - Registros (filas): {df.shape[0]}")
print(f"   - Variables (columnas): {df.shape[1]}")

a. Número de registros y variables:
   - Registros (filas): 27413
   - Variables (columnas): 19


### b. Tipo de dato de cada variable
**Descripción:** Detalla el tipo de dato asignado a cada columna (e.g., `object`, `int64`, `float64`) para asegurar su correcta interpretación analítica.

In [5]:
print("b. Tipo de dato de cada variable:")
for col in df.columns:
    print(f"   - {col}: {df[col].dtype}")

b. Tipo de dato de cada variable:
   - CODIGO: object
   - DISTRITO: object
   - DEPARTAMENTO: object
   - MUNICIPIO: object
   - ESTABLECIMIENTO: object
   - DIRECCION: object
   - TELEFONO: object
   - SUPERVISOR: object
   - DIRECTOR: object
   - NIVEL: object
   - SECTOR: object
   - AREA: object
   - STATUS: object
   - MODALIDAD: object
   - JORNADA: object
   - PLAN: object
   - DEPARTAMENTAL: object
   - DEPARTAMENTO_CONSULTADO: object
   - NIVEL_CONSULTADO: object


### c. Cantidad y porcentaje de valores faltantes por variable
**Descripción:** Cuantifica la presencia de valores nulos o ausentes en cada variable, calculando su proporción respecto al total de registros.

In [6]:
print("c. Cantidad y porcentaje de valores faltantes por variable:")
faltantes = df.isna().sum()
total = len(df)
for col in df.columns:
    nulos = faltantes[col]
    pct = (nulos / total) * 100
    print(f"   - {col}: {nulos} ({pct:.1f}%)")

c. Cantidad y porcentaje de valores faltantes por variable:
   - CODIGO: 0 (0.0%)
   - DISTRITO: 1616 (5.9%)
   - DEPARTAMENTO: 0 (0.0%)
   - MUNICIPIO: 0 (0.0%)
   - ESTABLECIMIENTO: 7 (0.0%)
   - DIRECCION: 279 (1.0%)
   - TELEFONO: 2800 (10.2%)
   - SUPERVISOR: 1620 (5.9%)
   - DIRECTOR: 4634 (16.9%)
   - NIVEL: 0 (0.0%)
   - SECTOR: 0 (0.0%)
   - AREA: 0 (0.0%)
   - STATUS: 0 (0.0%)
   - MODALIDAD: 0 (0.0%)
   - JORNADA: 0 (0.0%)
   - PLAN: 0 (0.0%)
   - DEPARTAMENTAL: 0 (0.0%)
   - DEPARTAMENTO_CONSULTADO: 0 (0.0%)
   - NIVEL_CONSULTADO: 0 (0.0%)


### d. Cantidad de valores únicos por variable
**Descripción:** Determina la cardinalidad de cada columna mostrando cuántos valores distintivos o categorías contiene.

In [7]:
print("d. Cantidad de valores únicos por variable:")
for col in df.columns:
    print(f"   - {col}: {df[col].nunique()}")

d. Cantidad de valores únicos por variable:
   - CODIGO: 27413
   - DISTRITO: 2017
   - DEPARTAMENTO: 23
   - MUNICIPIO: 357
   - ESTABLECIMIENTO: 10607
   - DIRECCION: 14724
   - TELEFONO: 13278
   - SUPERVISOR: 1501
   - DIRECTOR: 11459
   - NIVEL: 2
   - SECTOR: 4
   - AREA: 3
   - STATUS: 5
   - MODALIDAD: 2
   - JORNADA: 6
   - PLAN: 13
   - DEPARTAMENTAL: 26
   - DEPARTAMENTO_CONSULTADO: 23
   - NIVEL_CONSULTADO: 2


### e. Registros duplicados exactos
**Descripción:** Cuenta las filas que son idénticas en todas sus columnas para detectar redundancia total en la información.

In [8]:
duplicados_exactos = df.duplicated().sum()
print(f"e. Registros duplicados exactos: {duplicados_exactos}")

e. Registros duplicados exactos: 0


### f. Variables con valores fuera de dominio o inconsistentes
**Descripción:** Contrasta las opciones existentes en variables categóricas clave contra un conjunto predefinido de valores válidos esperados.

In [9]:
print("f. Variables con valores fuera de dominio o inconsistentes:")
dominios_esperados = {
    "DEPARTAMENTO": sorted(set(DEPARTAMENTOS) | {"ALTA VERAPAZ", "BAJA VERAPAZ", "CIUDAD CAPITAL"}),
    "NIVEL": ["BASICO", "DIVERSIFICADO", "PRIMARIA", "PARVULOS", "PREPRIMARIA BILINGUE", "PRIMARIA DE ADULTOS"],
    "SECTOR": ["OFICIAL", "PRIVADO", "MUNICIPAL", "COOPERATIVA"],
    "AREA": ["URBANA", "RURAL"],
    "STATUS": ["ABIERTA", "CERRADA DEFINITIVAMENTE", "CERRADA TEMPORALMENTE", "TEMPORAL NOMBRAMIENTO"],
    "MODALIDAD": ["MONOLINGUE", "BILINGUE"],
    "JORNADA": ["MATUTINA", "VESPERTINA", "NOCTURNA", "DOBLE", "SIN JORNADA"],
    "PLAN": ["DIARIO(REGULAR)", "SABATINO", "DOMINICAL", "FIN DE SEMANA", "IRREGULAR", "MIXTO", "A DISTANCIA", "VIRTUAL A DISTANCIA", "SEMIPRESENCIAL (FIN DE SEMANA)", "SEMIPRESENCIAL (UN DÍA A LA SEMANA)"]
}
for col, esperados in dominios_esperados.items():
    if col in df.columns:
        valores_reales = set(df[col].dropna().unique())
        fuera = valores_reales - set(esperados)
        if fuera:
            print(f"   - {col}: valores fuera del dominio esperado: {fuera}")
        else:
            print(f"   - {col}: todos los valores están dentro del dominio esperado.")

f. Variables con valores fuera de dominio o inconsistentes:
   - DEPARTAMENTO: todos los valores están dentro del dominio esperado.
   - NIVEL: todos los valores están dentro del dominio esperado.
   - SECTOR: todos los valores están dentro del dominio esperado.
   - AREA: valores fuera del dominio esperado: {'SIN ESPECIFICAR'}
   - STATUS: valores fuera del dominio esperado: {'TEMPORAL TITULOS'}
   - MODALIDAD: todos los valores están dentro del dominio esperado.
   - JORNADA: valores fuera del dominio esperado: {'INTERMEDIA'}
   - PLAN: valores fuera del dominio esperado: {'INTERCALADO', 'SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)', 'SEMIPRESENCIAL'}


### g. Variables con formatos inconsistentes
**Descripción:** Inspecciona la sintaxis de campos específicos (e.g., código institucional y teléfonos) para validar que cumplan con la estructura estandarizada.

In [10]:
print("g. Variables con formatos inconsistentes:")

# Evaluación del patrón de CODIGO
inconsistentes = []
if "CODIGO" in df.columns:
    codigos = df["CODIGO"].dropna().astype(str)
    patron = re.compile(r'^\d{2}-\d{2}-\d{4}-\d{2}$')
    inconsistentes = codigos[~codigos.str.match(patron)]
    if len(inconsistentes) > 0:
        print(f"   - CODIGO: {len(inconsistentes)} registros no siguen el formato ##-##-####-##")
        print(f"     Ejemplos: {inconsistentes.head(3).tolist()}")
    else:
        print("   - CODIGO: todos los códigos siguen el formato esperado.")

# Evaluación de TELEFONO
no_numericos = []
if "TELEFONO" in df.columns:
    telefonos = df["TELEFONO"].dropna().astype(str)
    limpios = telefonos.str.replace(r'[\s\-\(\)]', '', regex=True)
    no_numericos = limpios[~limpios.str.isdigit()]
    if len(no_numericos) > 0:
        print(f"   - TELEFONO: {len(no_numericos)} registros contienen caracteres no numéricos.")
        print(f"     Ejemplos: {no_numericos.head(3).tolist()}")
    else:
        print("   - TELEFONO: todos los valores son numéricos (tras limpieza).")

g. Variables con formatos inconsistentes:
   - CODIGO: todos los códigos siguen el formato esperado.
   - TELEFONO: 99 registros contienen caracteres no numéricos.
     Ejemplos: ['55074625,58722678', '22206556Y22516346', '22534814y22381691']


### h. Identificación de problemas potenciales de calidad de datos
**Descripción:** Sintetiza los hallazgos principales de las pruebas previas y consolida los riesgos potenciales encontrados en la base de datos.

In [11]:
print("h. Identificación de problemas potenciales de calidad de datos:")
problemas = []

altos_faltantes = [col for col in df.columns if (df[col].isna().sum()/len(df))*100 > 50]
if altos_faltantes:
    problemas.append(f"   - Columnas con más del 50% de datos faltantes: {altos_faltantes}")
if duplicados_exactos > 0:
    problemas.append(f"   - Existen {duplicados_exactos} filas duplicadas exactas.")
if any(fuera for col, fuera in dominios_esperados.items() if col in df.columns):
    problemas.append("   - Algunas variables categóricas tienen valores fuera del dominio esperado (ver sección f).")
if "CODIGO" in df.columns and len(inconsistentes) > 0:
    problemas.append(f"   - {len(inconsistentes)} códigos tienen formato inesperado.")
if "TELEFONO" in df.columns and len(no_numericos) > 0:
    problemas.append(f"   - {len(no_numericos)} teléfonos contienen caracteres no numéricos.")

if problemas:
    for p in problemas:
        print(p)
else:
    print("   - No se detectaron problemas graves de calidad de datos.")

h. Identificación de problemas potenciales de calidad de datos:
   - Algunas variables categóricas tienen valores fuera del dominio esperado (ver sección f).
   - 99 teléfonos contienen caracteres no numéricos.


## 5. Plan de Limpieza

### 5.1 Resumen

Dataset crudo: 27,413 filas, 19 variables, todas tipo object. 0 duplicados exactos. Problemas principales: NA reales y disfrazados (".", "-", "SIN DATO"), formatos inconsistentes en TELEFONO y DISTRITO, categorías que el diagnóstico marcó como "fuera de dominio" pero en realidad son válidas (SIN ESPECIFICAR, INTERMEDIA, etc.), 2 columnas redundantes (DEPARTAMENTO_CONSULTADO, NIVEL_CONSULTADO), y posibles duplicados parciales en ESTABLECIMIENTO (mismo plantel con BASICO/DIVERSIFICADO bajo distinto CODIGO).

Principio general: no se tocan tildes ni ortografía de nombres/direcciones; solo espacios, placeholders y puntuación mal usada.

### 5.2 Variables prioritarias

TELEFONO, DIRECTOR, ESTABLECIMIENTO, DIRECCION y DISTRITO son las que más limpieza requieren (ver detalle en 5.3). MUNICIPIO necesita validación contra catálogo; AREA/STATUS/JORNADA/PLAN solo necesitan ampliar el dominio de referencia; DEPARTAMENTO_CONSULTADO y NIVEL_CONSULTADO se eliminarán por ser redundantes.

### 5.3 División del trabajo

Se dividen las 19 columnas entre los 3 integrantes del grupo. Carga desigual a propósito: Olivier ya avanzó más trabajo antes (scraping y unificación de los datos), así que recibe menos columnas.

| Integrante | Columnas asignadas | Cantidad |
|---|---|---|
| Erick | ESTABLECIMIENTO, DIRECCION, TELEFONO, SUPERVISOR, DIRECTOR, NIVEL, DISTRITO, MUNICIPIO | 8 |
| Fabián | SECTOR, AREA, STATUS, MODALIDAD, JORNADA, PLAN, DEPARTAMENTAL | 7 |
| Olivier | CODIGO, DEPARTAMENTO, DEPARTAMENTO_CONSULTADO, NIVEL_CONSULTADO | 4 |

Cada quien documenta sus transformaciones en la tabla de registro de transformaciones (sección 6).

### 5.4 Estrategia por variable

| Variable | Problema | Regla | Riesgo |
|---|---|---|---|
| CODIGO | Ninguno; formato 100% válido | Mantener como texto; validar con regex | Perder ceros a la izquierda si se numeriza |
| DISTRITO | Formato inconsistente (3/6/10 caracteres) | strip(); documentar variantes; NA si falta | Inventar un código al forzar un formato único |
| DEPARTAMENTO | Ninguno | strip(); convertir a category | Confundir CIUDAD CAPITAL con GUATEMALA |
| MUNICIPIO | "ZONA N" en Ciudad Capital en vez de municipio | Validar contra catálogo INE; marcar zonas con bandera | Perder granularidad si se fuerza a "GUATEMALA" |
| DEPARTAMENTAL | Un departamento puede tener varias oficinas | strip(); category; documentar diferencia con DEPARTAMENTO | Confundirla con el departamento geográfico |
| DEPARTAMENTO_CONSULTADO / NIVEL_CONSULTADO | 100% redundantes | Eliminar | Perder trazabilidad de la extracción (bajo) |
| ESTABLECIMIENTO | Puntuación mal usada (comilla invertida, guion bajo); posibles duplicados parciales | strip(); corregir puntuación solo en casos puntuales; fuzzy-match para detectar duplicados sin eliminar | Dañar ortografía maya o fusionar escuelas distintas |
| DIRECCION | 1% NA + placeholders; 26 registros no en mayúsculas | Placeholders exactos a NA; homologar mayúsculas; strip() | Borrar un "-" que sea parte real de la dirección |
| SUPERVISOR | 5.9% NA | strip(); similitud solo para detectar errores de tecleo | Fusionar dos personas distintas por error |
| DIRECTOR | 16.9% NA + 95 placeholders | Placeholders exactos a NA; strip() | Igual que SUPERVISOR |
| TELEFONO | Formatos dispares; 84 celdas con más de un número | Derivar TELEFONO_PRINCIPAL/LISTA; mantener como texto | Separador no detectado deja el caso sin dividir |
| NIVEL / NIVEL_CONSULTADO | Ninguno | strip(); category | — |
| SECTOR | Ninguno | strip(); category | — |
| AREA | "SIN ESPECIFICAR" fuera del dominio inicial del diagnóstico | Agregar como categoría válida | Confundirla con NA |
| STATUS | "TEMPORAL TITULOS" fuera del dominio inicial | Agregar como categoría válida | Confundir con TEMPORAL NOMBRAMIENTO |
| MODALIDAD | Ninguno | strip(); category | — |
| JORNADA | "INTERMEDIA" fuera del dominio inicial | Agregar como categoría válida | Confundir con DOBLE |
| PLAN | 3 categorías fuera del dominio inicial | Ampliar catálogo a las 13 categorías reales | Fusionar por error variantes de SEMIPRESENCIAL

## 6. Limpieza de Datos
#### Fabián

In [12]:
registro_transformaciones = []

def registrar(variable, problema, transformacion, registros_afectados, justificacion):
    registro_transformaciones.append({
        "Problema detectado": problema,
        "Variable": variable,
        "Transformación": transformacion,
        "Registros afectados": int(registros_afectados),
        "Justificación": justificacion,
    })

#### SECTOR y MODALIDAD


In [13]:
for col, dominio in [("SECTOR", ["OFICIAL", "PRIVADO", "MUNICIPAL", "COOPERATIVA"]),
                      ("MODALIDAD", ["MONOLINGUE", "BILINGUE"])]:
    antes = df[col].copy()
    df[col] = df[col].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
    afectados = int((df[col] != antes).sum())
    fuera_de_dominio = df.loc[~df[col].isin(dominio), col]
    df[col] = pd.Categorical(df[col], categories=dominio)

    registrar(
        variable=col,
        problema="Ninguno",
        transformacion="strip() + upper() + category",
        registros_afectados=afectados,
        justificacion="Dominio ya validado; solo se estandariza formato.",
    )
    print(f"{col}: {afectados} registros modificados por formato, {len(fuera_de_dominio)} fuera de dominio, dtype = {df[col].dtype}")

SECTOR: 0 registros modificados por formato, 0 fuera de dominio, dtype = category
MODALIDAD: 0 registros modificados por formato, 0 fuera de dominio, dtype = category


#### AREA


In [14]:
dominio_area = ["URBANA", "RURAL", "SIN ESPECIFICAR"]

antes = df["AREA"].copy()
df["AREA"] = df["AREA"].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
afectados_formato = int((df["AREA"] != antes).sum())
afectados_dominio = int((df["AREA"] == "SIN ESPECIFICAR").sum())
df["AREA"] = pd.Categorical(df["AREA"], categories=dominio_area)

registrar(
    variable="AREA",
    problema="'SIN ESPECIFICAR' fuera del dominio inicial",
    transformacion="Se agrega como categoría válida",
    registros_afectados=afectados_dominio,
    justificacion="Es una categoría real del MINEDUC, no un vacío.",
)
print(f"AREA: {afectados_formato} registros modificados por formato, {afectados_dominio} con SIN ESPECIFICAR, dtype = {df['AREA'].dtype}")
print(df["AREA"].value_counts(dropna=False))

AREA: 0 registros modificados por formato, 5 con SIN ESPECIFICAR, dtype = category
AREA
URBANA             18779
RURAL               8629
SIN ESPECIFICAR        5
Name: count, dtype: int64


#### STATUS

In [15]:
dominio_status = ["ABIERTA", "CERRADA DEFINITIVAMENTE", "CERRADA TEMPORALMENTE", "TEMPORAL NOMBRAMIENTO", "TEMPORAL TITULOS"]

antes = df["STATUS"].copy()
df["STATUS"] = df["STATUS"].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
afectados_formato = int((df["STATUS"] != antes).sum())
afectados_dominio = int((df["STATUS"] == "TEMPORAL TITULOS").sum())
df["STATUS"] = pd.Categorical(df["STATUS"], categories=dominio_status)

registrar(
    variable="STATUS",
    problema="'TEMPORAL TITULOS' fuera del dominio inicial",
    transformacion="Se agrega como categoría válida",
    registros_afectados=afectados_dominio,
    justificacion="Autorización real del MINEDUC, no error de captura.",
)
print(f"STATUS: {afectados_formato} registros modificados por formato, {afectados_dominio} con TEMPORAL TITULOS, dtype = {df['STATUS'].dtype}")
print(df["STATUS"].value_counts(dropna=False))

STATUS: 0 registros modificados por formato, 110 con TEMPORAL TITULOS, dtype = category
STATUS
ABIERTA                    16830
CERRADA TEMPORALMENTE       6292
CERRADA DEFINITIVAMENTE     4067
TEMPORAL NOMBRAMIENTO        114
TEMPORAL TITULOS             110
Name: count, dtype: int64


#### JORNADA


In [16]:
dominio_jornada = ["MATUTINA", "VESPERTINA", "NOCTURNA", "DOBLE", "SIN JORNADA", "INTERMEDIA"]

antes = df["JORNADA"].copy()
df["JORNADA"] = df["JORNADA"].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
afectados_formato = int((df["JORNADA"] != antes).sum())
afectados_dominio = int((df["JORNADA"] == "INTERMEDIA").sum())
df["JORNADA"] = pd.Categorical(df["JORNADA"], categories=dominio_jornada)

registrar(
    variable="JORNADA",
    problema="'INTERMEDIA' fuera del dominio inicial",
    transformacion="Se agrega como categoría válida",
    registros_afectados=afectados_dominio,
    justificacion="Jornada real del MINEDUC, no un error de captura.",
)
print(f"JORNADA: {afectados_formato} registros modificados por formato, {afectados_dominio} con INTERMEDIA, dtype = {df['JORNADA'].dtype}")
print(df["JORNADA"].value_counts(dropna=False))

JORNADA: 0 registros modificados por formato, 205 con INTERMEDIA, dtype = category
JORNADA
VESPERTINA     10035
MATUTINA        7347
DOBLE           6758
SIN JORNADA     2106
NOCTURNA         962
INTERMEDIA       205
Name: count, dtype: int64


#### PLAN

3 categorías ("SEMIPRESENCIAL", "SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)", "INTERCALADO")

In [17]:
dominio_plan_inicial = ["DIARIO(REGULAR)", "SABATINO", "DOMINICAL", "FIN DE SEMANA", "IRREGULAR", "MIXTO",
                         "A DISTANCIA", "VIRTUAL A DISTANCIA", "SEMIPRESENCIAL (FIN DE SEMANA)",
                         "SEMIPRESENCIAL (UN DÍA A LA SEMANA)"]
categorias_nuevas = ["SEMIPRESENCIAL", "SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)", "INTERCALADO"]
dominio_plan = dominio_plan_inicial + categorias_nuevas

antes = df["PLAN"].copy()
df["PLAN"] = df["PLAN"].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
afectados_formato = int((df["PLAN"] != antes).sum())
afectados_dominio = int(df["PLAN"].isin(categorias_nuevas).sum())
df["PLAN"] = pd.Categorical(df["PLAN"], categories=dominio_plan)

registrar(
    variable="PLAN",
    problema="3 categorías fuera del dominio inicial",
    transformacion="Se amplía el catálogo a 13 categorías reales",
    registros_afectados=afectados_dominio,
    justificacion="Son modalidades reales del MINEDUC, no errores de captura.",
)
print(f"PLAN: {afectados_formato} registros modificados por formato, {afectados_dominio} en categorías nuevas, dtype = {df['PLAN'].dtype}")
print(df["PLAN"].value_counts(dropna=False))

PLAN: 0 registros modificados por formato, 357 en categorías nuevas, dtype = category
PLAN
DIARIO(REGULAR)                          18870
FIN DE SEMANA                             5454
SEMIPRESENCIAL (FIN DE SEMANA)            1032
SEMIPRESENCIAL (UN DÍA A LA SEMANA)        843
A DISTANCIA                                512
SEMIPRESENCIAL                             220
SABATINO                                   139
SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)      133
VIRTUAL A DISTANCIA                        127
DOMINICAL                                   64
MIXTO                                        8
IRREGULAR                                    7
INTERCALADO                                  4
Name: count, dtype: int64


#### DEPARTAMENTAL


In [18]:
antes = df["DEPARTAMENTAL"].copy()
df["DEPARTAMENTAL"] = df["DEPARTAMENTAL"].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
afectados = int((df["DEPARTAMENTAL"] != antes).sum())
dominio_departamental = sorted(df["DEPARTAMENTAL"].dropna().unique())
df["DEPARTAMENTAL"] = pd.Categorical(df["DEPARTAMENTAL"], categories=dominio_departamental)

registrar(
    variable="DEPARTAMENTAL",
    problema="Ninguno; un mismo DEPARTAMENTO puede tener varias DEPARTAMENTAL",
    transformacion="strip() + upper() + category",
    registros_afectados=afectados,
    justificacion="Representa la oficina administrativa del MINEDUC, no el departamento geográfico.",
)
print(f"DEPARTAMENTAL: {afectados} registros modificados por formato, {len(dominio_departamental)} categorías, dtype = {df['DEPARTAMENTAL'].dtype}")

DEPARTAMENTAL: 0 registros modificados por formato, 26 categorías, dtype = category


#### ESTABLECIMIENTO


In [19]:
antes = df["ESTABLECIMIENTO"].copy()
df["ESTABLECIMIENTO"] = df["ESTABLECIMIENTO"].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
# Acronimos mal delimitados con guion bajo en vez de parentesis, p.ej. "..._CEPMEM" -> "... (CEPMEM)"
df["ESTABLECIMIENTO"] = df["ESTABLECIMIENTO"].str.replace(r"_+([A-Z]+)_*$", r" (\1)", regex=True)
df["ESTABLECIMIENTO"] = df["ESTABLECIMIENTO"].str.replace(r"\s+", " ", regex=True).str.strip()
afectados = int((df["ESTABLECIMIENTO"] != antes).sum())

registrar(
    variable="ESTABLECIMIENTO",
    problema="Espacios/mayusculas inconsistentes y siglas delimitadas con guion bajo en vez de parentesis",
    transformacion="strip() + upper() + colapsar espacios + envolver siglas sueltas entre parentesis en vez de guion bajo",
    registros_afectados=afectados,
    justificacion="Estandariza formato sin alterar ortografia de nombres propios ni de idiomas mayas (no se tocan tildes, apostrofes ni saltillos).",
)
print(f"ESTABLECIMIENTO: {afectados} registros modificados por formato, {df['ESTABLECIMIENTO'].nunique()} valores unicos")


ESTABLECIMIENTO: 10 registros modificados por formato, 10608 valores unicos


#### ESTABLECIMIENTO - Duplicados parciales

Se buscan posibles duplicados parciales combinando: mismo `MUNICIPIO`, similitud de nombre (RapidFuzz `token_sort_ratio` >= 90) y misma `DIRECCION`. No se elimina ni fusiona ningun registro automaticamente; se agrupan en componentes conexas para revision manual.


In [21]:
from rapidfuzz import fuzz, process

_nombres_norm = df["ESTABLECIMIENTO"].astype(str)
_municipio_norm = df["MUNICIPIO"].astype(str).str.strip().str.upper()
_direccion_norm = df["DIRECCION"].astype(str).str.strip().str.upper()

pares_candidatos = []
for _mun, _idx in df.groupby(_municipio_norm).groups.items():
    _idx = list(_idx)
    if len(_idx) < 2:
        continue
    _nombres = _nombres_norm.loc[_idx].tolist()
    _mat = process.cdist(_nombres, _nombres, scorer=fuzz.token_sort_ratio)
    _n = len(_nombres)
    for _i in range(_n):
        for _j in range(_i + 1, _n):
            if _mat[_i, _j] < 90 or _nombres[_i] == _nombres[_j]:
                continue
            _a, _b = _idx[_i], _idx[_j]
            if _direccion_norm.loc[_a] != _direccion_norm.loc[_b] or _direccion_norm.loc[_a] in ("NAN", "-", ".", ""):
                continue
            pares_candidatos.append((_a, _b))

# Componentes conexas (union-find) para agrupar cadenas de posibles duplicados
_parent = {}
def _find(x):
    _parent.setdefault(x, x)
    while _parent[x] != x:
        _parent[x] = _parent[_parent[x]]
        x = _parent[x]
    return x
def _union(a, b):
    ra, rb = _find(a), _find(b)
    if ra != rb:
        _parent[ra] = rb

for _a, _b in pares_candidatos:
    _union(_a, _b)

_grupos = {}
for _nodo in _parent:
    _raiz = _find(_nodo)
    _grupos.setdefault(_raiz, []).append(_nodo)

filas_grupo = []
for _gid, _idxs in enumerate(_grupos.values(), start=1):
    for _i in _idxs:
        filas_grupo.append({"GRUPO_DUPLICADO": _gid, "CODIGO": df.loc[_i, "CODIGO"], "ESTABLECIMIENTO": df.loc[_i, "ESTABLECIMIENTO"],
                             "NIVEL": df.loc[_i, "NIVEL"], "DIRECCION": df.loc[_i, "DIRECCION"], "MUNICIPIO": df.loc[_i, "MUNICIPIO"]})

posibles_duplicados_establecimiento = pd.DataFrame(filas_grupo).sort_values(["GRUPO_DUPLICADO"])

registrar(
    variable="ESTABLECIMIENTO",
    problema="Posibles duplicados parciales (mismo plantel listado varias veces, p.ej. una fila por NIVEL)",
    transformacion="Deteccion por similitud de cadena con RapidFuzz, mas misma direccion; SIN eliminar ni fusionar registros",
    registros_afectados=len(_parent),
    justificacion="No hay certeza suficiente para fusionar automaticamente; se documentan para revision manual del equipo.",
)
print(f"Grupos de posibles duplicados: {len(_grupos)} | Registros involucrados: {len(_parent)} ({len(_parent)/len(df)*100:.1f}%)")
posibles_duplicados_establecimiento.head(15)


Grupos de posibles duplicados: 753 | Registros involucrados: 2766 (10.1%)


,GRUPO_DUPLICADO,CODIGO,ESTABLECIMIENTO,NIVEL,DIRECCION,MUNICIPIO
0,1,22-04-0030-46,COLEGIO PARTICULAR MIXTO AGUABLANQUENSE,DIVERSIFICADO,BARRIO LAS CASITAS,AGUA BLANCA
1,1,22-04-2318-45,INEB COLEGIO PARTICULAR MIXTO AGUABLANQUENSE,BASICO,BARRIO LAS CASITAS,AGUA BLANCA
2,2,13-27-0067-46,INSTITUTO NACIONAL DE EDUCACIÓN DIVERSIFICADA,DIVERSIFICADO,ALDEA CANTZELA,AGUACATAN
3,2,13-27-0090-46,INED INSTITUTO NACIONAL DE EDUCACION DIVERSIFI...,DIVERSIFICADO,ALDEA CANTZELA,AGUACATAN
4,3,13-27-0058-46,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,DIVERSIFICADO,ALDEA CLIMENTORO,AGUACATAN
5,3,13-27-0091-46,INED INSTITUTO NACIONAL DE EDUCACION DIVERSIFI...,DIVERSIFICADO,ALDEA CLIMENTORO,AGUACATAN
7,4,01-14-8051-46,LICEO MIXTO CRISTIANO 'JIREH',DIVERSIFICADO,"3A. AVENIDA 2-45, COLONIA VILLA ALBORADA",AMATITLAN
8,4,01-14-0140-45,"LICEO MIXTO CRISTIANO ""JIREH""",BASICO,"3A. AVENIDA 2-45, COLONIA VILLA ALBORADA",AMATITLAN
6,4,01-14-0141-46,"LICEO MIXTO CRISTIANO ""JIREH""",DIVERSIFICADO,"3A. AVENIDA 2-45, COLONIA VILLA ALBORADA",AMATITLAN
9,5,01-14-0185-46,LICEO APLICADO EN COMPUTACION DE AMATITLAN,DIVERSIFICADO,5A. AVENIDA 6-86,AMATITLAN


#### DIRECCION


In [ ]:
placeholders_direccion = ["-", ".", "..", "...", "N/A", "NA", "NULL", "SIN DATO", "NINGUNA", "NO TIENE", "S/D"]

antes = df["DIRECCION"].copy()
_limpio = df["DIRECCION"].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
_limpio = _limpio.mask(df["DIRECCION"].isna())
afectados_formato = int(((_limpio != antes) & df["DIRECCION"].notna()).sum())

_es_placeholder = _limpio.isin(placeholders_direccion)
afectados_placeholder = int(_es_placeholder.sum())
_limpio[_es_placeholder] = pd.NA
df["DIRECCION"] = _limpio

registrar(
    variable="DIRECCION",
    problema="Placeholders de dato faltante ('-', '.', etc.) y mayusculas/espacios inconsistentes",
    transformacion="strip() + upper() + colapsar espacios; placeholders exactos -> NA",
    registros_afectados=afectados_placeholder + afectados_formato,
    justificacion="Los placeholders no aportan informacion real; se homologa formato sin alterar el contenido de la direccion.",
)
print(f"DIRECCION: {afectados_placeholder} placeholders convertidos a NA, {afectados_formato} normalizados por formato, NA totales = {df['DIRECCION'].isna().sum()}")


DIRECCION: 10 placeholders convertidos a NA, 26 normalizados por formato, NA totales = 289


#### TELEFONO

Se mantiene la columna original como texto (formatos declarados demasiado heterogeneos para forzar un unico patron: numeros multiples separados por coma/'Y', extensiones, FAX, CEL). Se agregan dos variables derivadas:
- `TELEFONO_PRINCIPAL`: primer numero de 6 a 8 digitos encontrado.
- `TELEFONO_LISTA`: todos los numeros de 6 a 8 digitos encontrados, separados por '; '.

No se intenta expandir rangos abreviados (p.ej. "77636578 AL 80") por el riesgo de inventar numeros que no existen.


In [ ]:
import re

_patron_telefono = re.compile(r"\d{3,4}-\d{4}|\d{6,8}")

def _extraer_telefonos(valor):
    if pd.isna(valor):
        return []
    return [m.replace("-", "") for m in _patron_telefono.findall(str(valor))]

_listas = df["TELEFONO"].apply(_extraer_telefonos)
df["TELEFONO_PRINCIPAL"] = _listas.apply(lambda l: l[0] if l else pd.NA)
df["TELEFONO_LISTA"] = _listas.apply(lambda l: "; ".join(l) if l else pd.NA)

afectados = int(df["TELEFONO"].notna().sum())
sin_numero_extraible = int((df["TELEFONO"].notna() & df["TELEFONO_PRINCIPAL"].isna()).sum())
formato_corto = int((df["TELEFONO_PRINCIPAL"].str.len() < 7).sum()) if df["TELEFONO_PRINCIPAL"].notna().any() else 0

registrar(
    variable="TELEFONO",
    problema="Formatos heterogeneos: varios numeros por celda, guiones internos, etiquetas FAX/CEL/EXT, rangos abreviados",
    transformacion="Se crean TELEFONO_PRINCIPAL y TELEFONO_LISTA (variables derivadas) extrayendo numeros de 6-8 digitos; TELEFONO original se conserva intacto",
    registros_afectados=afectados,
    justificacion="Preserva el dato crudo (auditable) y entrega un campo utilizable para analisis/validacion sin inventar informacion en los casos ambiguos.",
)
print(f"TELEFONO: {afectados} celdas no vacias procesadas, {sin_numero_extraible} sin numero extraible, {formato_corto} con menos de 7 digitos (posible numero antiguo, no se descarta)")


TELEFONO: 24613 celdas no vacias procesadas, 13 sin numero extraible, 20 con menos de 7 digitos (posible numero antiguo, no se descarta)


#### TILDES


In [ ]:
import unicodedata

def quitar_tildes(texto):
    return "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")

def unificar_variantes_de_tilde(serie_limpia):
    clave = serie_limpia.apply(lambda s: quitar_tildes(s) if pd.notna(s) else s)
    canonico = serie_limpia.groupby(clave).transform(lambda s: s.value_counts().idxmax())
    return canonico


#### SUPERVISOR


In [ ]:
antes = df["SUPERVISOR"].copy()
_limpio = df["SUPERVISOR"].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
_limpio = _limpio.mask(df["SUPERVISOR"].isna())
afectados_formato = int(((_limpio != antes) & df["SUPERVISOR"].notna()).sum())

_canonico = unificar_variantes_de_tilde(_limpio.dropna())
afectados_tilde = int((_canonico != _limpio.loc[_canonico.index]).sum())
_limpio.loc[_canonico.index] = _canonico
df["SUPERVISOR"] = _limpio

registrar(
    variable="SUPERVISOR",
    problema="Mismo supervisor capturado con y sin tildes, genera categorias duplicadas",
    transformacion="strip() + upper() + colapsar espacios; unificacion de variantes por tilde a la forma mas frecuente",
    registros_afectados=afectados_formato + afectados_tilde,
    justificacion="Es la misma persona escrita de dos formas distintas; unificar reduce categorias falsas sin fusionar personas diferentes (se valida por nombre completo).",
)
print(f"SUPERVISOR: {afectados_formato} por formato, {afectados_tilde} unificados por tilde, {df['SUPERVISOR'].nunique()} valores unicos")


SUPERVISOR: 0 por formato, 2611 unificados por tilde, 1265 valores unicos


#### DIRECTOR


In [ ]:
placeholders_director = ["-", "--", "---", "----", "-----", "------", ".", "..", "...", ",",
                         "SIN DATO", "S/D", "X", "XXX", "XXXXXX", "0", "000000"]

antes = df["DIRECTOR"].copy()
_limpio = df["DIRECTOR"].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
_limpio = _limpio.mask(df["DIRECTOR"].isna())
_es_placeholder = _limpio.isin(placeholders_director)
afectados_placeholder = int(_es_placeholder.sum())
_limpio[_es_placeholder] = pd.NA

_canonico = unificar_variantes_de_tilde(_limpio.dropna())
afectados_tilde = int((_canonico != _limpio.loc[_canonico.index]).sum())
_limpio.loc[_canonico.index] = _canonico
df["DIRECTOR"] = _limpio

registrar(
    variable="DIRECTOR",
    problema="Placeholders de dato faltante, como '-', '.', 'SIN DATO', etc., y categorias duplicadas por tildes",
    transformacion="strip() + upper() + colapsar espacios; placeholders -> NA; unificacion de variantes por tilde",
    registros_afectados=afectados_placeholder + afectados_tilde,
    justificacion="Mismo criterio que SUPERVISOR: separa dato realmente faltante de nombre real, y evita contar la misma persona dos veces por tildes.",
)
print(f"DIRECTOR: {afectados_placeholder} placeholders -> NA, {afectados_tilde} unificados por tilde, NA totales = {df['DIRECTOR'].isna().sum()}")


DIRECTOR: 626 placeholders -> NA, 377 unificados por tilde, NA totales = 5260


#### NIVEL


In [ ]:
dominio_nivel = ["BASICO", "DIVERSIFICADO"]

antes = df["NIVEL"].copy()
df["NIVEL"] = df["NIVEL"].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
afectados = int((df["NIVEL"] != antes).sum())
df["NIVEL"] = pd.Categorical(df["NIVEL"], categories=dominio_nivel)

registrar(
    variable="NIVEL",
    problema="Ninguno; dominio ya validado en el diagnostico",
    transformacion="strip() + upper() + category",
    registros_afectados=afectados,
    justificacion="Solo se estandariza formato y tipo de dato.",
)
print(f"NIVEL: {afectados} registros modificados por formato, dtype = {df['NIVEL'].dtype}")
print(df["NIVEL"].value_counts(dropna=False))


NIVEL: 0 registros modificados por formato, dtype = category
NIVEL
BASICO           15546
DIVERSIFICADO    11867
Name: count, dtype: int64


#### DISTRITO

Se identificaron dos formatos validos en los datos: `DD-NNN` (departamento-distrito, 6 caracteres) y `DD-MM-NNNN` (departamento-municipio-secuencial, 10 caracteres). Los valores de 3 caracteres (p.ej. "01-") no son un tercer formato valido, sino codigos truncados/incompletos a los que les falta el sufijo numerico; como no se puede reconstruir el dato faltante, se marcan como NA en vez de inventarlo.


In [ ]:
patron_distrito_6 = r"^\d{2}-\d{3}$"
patron_distrito_10 = r"^\d{2}-\d{2}-\d{4}$"

antes = df["DISTRITO"].copy()
_limpio = df["DISTRITO"].astype(str).str.strip()
_limpio = _limpio.mask(df["DISTRITO"].isna())
_valido = _limpio.isna() | _limpio.str.match(patron_distrito_6) | _limpio.str.match(patron_distrito_10)
afectados_incompleto = int((~_valido).sum())
_limpio[~_valido] = pd.NA
df["DISTRITO"] = _limpio

registrar(
    variable="DISTRITO",
    problema="Codigos truncados de 3 caracteres (p.ej. '01-') sin el sufijo numerico",
    transformacion="Se mantienen los dos formatos validos, DD-NNN y DD-MM-NNNN, como texto; codigos incompletos -> NA",
    registros_afectados=afectados_incompleto,
    justificacion="El sufijo faltante no puede reconstruirse de forma confiable; se prefiere marcar como faltante a inventar un valor.",
)
print(f"DISTRITO: {afectados_incompleto} codigos incompletos convertidos a NA, dtype = {df['DISTRITO'].dtype}")


DISTRITO: 75 codigos incompletos convertidos a NA, dtype = str


#### MUNICIPIO

Se detectaron 4,155 registros (todos en `DEPARTAMENTO == "CIUDAD CAPITAL"`) donde `MUNICIPIO` contiene "ZONA N" en vez de un nombre de municipio real. Esto ocurre porque, administrativamente, Ciudad Capital corresponde a un unico municipio (Guatemala) dividido en zonas; el sistema del MINEDUC reporta la zona en el campo de municipio. No se sobrescribe el valor original (se perderia informacion de ubicacion mas fina); en su lugar se agrega una bandera para dejar el caso documentado y disponible para la validacion de dominio en la seccion 7.


In [ ]:
antes = df["MUNICIPIO"].copy()
df["MUNICIPIO"] = df["MUNICIPIO"].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
afectados_formato = int((df["MUNICIPIO"] != antes).sum())

df["MUNICIPIO_ES_ZONA_CAPITALINA"] = df["MUNICIPIO"].str.match(r"^ZONA\s*\d+$")
afectados_zona = int(df["MUNICIPIO_ES_ZONA_CAPITALINA"].sum())

registrar(
    variable="MUNICIPIO",
    problema="'ZONA N' en vez de nombre de municipio para los registros de Ciudad Capital",
    transformacion="strip() + upper() + colapsar espacios; se agrega bandera derivada MUNICIPIO_ES_ZONA_CAPITALINA (no se sobrescribe el valor original)",
    registros_afectados=afectados_formato + afectados_zona,
    justificacion="Ciudad Capital no tiene municipios en el sentido tradicional; forzar el valor a 'GUATEMALA' perderia la zona. Queda documentado para la validacion de catalogo en la seccion 7 (pendiente validar catalogo INE completo para el resto de municipios).",
)
print(f"MUNICIPIO: {afectados_formato} por formato, {afectados_zona} marcados como zona capitalina, {df['MUNICIPIO'].nunique()} valores unicos")


MUNICIPIO: 0 por formato, 4155 marcados como zona capitalina, 357 valores unicos


#### Verificacion rapida, columnas de Erick

In [ ]:
columnas_limpiadas_erick = ["ESTABLECIMIENTO", "DIRECCION", "SUPERVISOR", "DIRECTOR", "NIVEL", "DISTRITO", "MUNICIPIO"]
for col in columnas_limpiadas_erick:
    print(f"{col}: dtype={df[col].dtype}, nulos={df[col].isna().sum()} ({df[col].isna().mean()*100:.1f}%), unicos={df[col].nunique()}")
print("\nTELEFONO_PRINCIPAL: dtype=", df["TELEFONO_PRINCIPAL"].dtype, ", nulos=", df["TELEFONO_PRINCIPAL"].isna().sum())
print("TELEFONO_LISTA: dtype=", df["TELEFONO_LISTA"].dtype, ", nulos=", df["TELEFONO_LISTA"].isna().sum())


ESTABLECIMIENTO: dtype=str, nulos=7 (0.0%), unicos=10607
DIRECCION: dtype=str, nulos=289 (1.1%), unicos=14719
SUPERVISOR: dtype=str, nulos=1620 (5.9%), unicos=1265
DIRECTOR: dtype=str, nulos=5260 (19.2%), unicos=11168
NIVEL: dtype=category, nulos=0 (0.0%), unicos=2
DISTRITO: dtype=str, nulos=1691 (6.2%), unicos=2013
MUNICIPIO: dtype=str, nulos=0 (0.0%), unicos=357

TELEFONO_PRINCIPAL: dtype= str , nulos= 2813
TELEFONO_LISTA: dtype= str , nulos= 2813


#### Olivier


In [ ]:
patron_codigo = re.compile(r'^\d{2}-\d{2}-\d{4}-\d{2}$')

antes = df["CODIGO"].copy()
df["CODIGO"] = df["CODIGO"].astype(str).str.strip()
afectados_formato = int((df["CODIGO"] != antes).sum())

no_valido = ~df["CODIGO"].str.match(patron_codigo)
afectados_invalidos = int(no_valido.sum())
if afectados_invalidos > 0:
    print(f"Advertencia: {afectados_invalidos} códigos no siguen el patrón ##-##-####-## tras strip().")

registrar(
    variable="CODIGO",
    problema="Ninguno; el diagnóstico (4g) confirmó que el 100% de los códigos siguen el formato ##-##-####-##",
    transformacion="strip(); se mantiene como texto (no se numeriza) para no perder ceros a la izquierda",
    registros_afectados=afectados_formato,
    justificacion="Al no haber problema real, solo se aplica una limpieza preventiva de espacios; convertir CODIGO a numérico arriesgaría perder ceros a la izquierda con formato administrativo."
)


In [ ]:
dominio_departamento = sorted(DEPARTAMENTOS)

antes = df["DEPARTAMENTO"].copy()
df["DEPARTAMENTO"] = df["DEPARTAMENTO"].astype(str).str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
afectados_formato = int((df["DEPARTAMENTO"] != antes).sum())

fuera_de_dominio = df.loc[~df["DEPARTAMENTO"].isin(dominio_departamento), "DEPARTAMENTO"]
if len(fuera_de_dominio) > 0:
    print(f"Advertencia: {len(fuera_de_dominio)} valores de DEPARTAMENTO fuera del dominio esperado: {fuera_de_dominio.unique()}")

df["DEPARTAMENTO"] = pd.Categorical(df["DEPARTAMENTO"], categories=dominio_departamento)

registrar(
    variable="DEPARTAMENTO",
    problema="Ninguno; todos los valores están dentro del dominio esperado (incluye CIUDAD CAPITAL como departamento propio)",
    transformacion="strip() + upper() + category",
    registros_afectados=afectados_formato,
    justificacion="CIUDAD CAPITAL se mantiene distinto de GUATEMALA porque así lo reporta el MINEDUC administrativamente; fusionarlos perdería esa distinción real."
)


In [ ]:
# Verificación de redundancia antes de eliminar: DEPARTAMENTO_CONSULTADO / NIVEL_CONSULTADO
# deberían coincidir (salvo diferencias de formato) con DEPARTAMENTO y NIVEL respectivamente,
# ya que eran los parámetros usados para pedir cada descarga del scraping.
if "DEPARTAMENTO_CONSULTADO" in df.columns:
    coincide_depto = (
        df["DEPARTAMENTO_CONSULTADO"].astype(str).str.strip().str.upper() == df["DEPARTAMENTO"].astype(str)
    ).mean()
    print(f"Coincidencia DEPARTAMENTO_CONSULTADO vs DEPARTAMENTO: {coincide_depto:.1%}")

if "NIVEL_CONSULTADO" in df.columns:
    coincide_nivel = (
        df["NIVEL_CONSULTADO"].astype(str).str.strip().str.upper() == df["NIVEL"].astype(str)
    ).mean()
    print(f"Coincidencia NIVEL_CONSULTADO vs NIVEL: {coincide_nivel:.1%}")

columnas_redundantes = [c for c in ["DEPARTAMENTO_CONSULTADO", "NIVEL_CONSULTADO"] if c in df.columns]
df = df.drop(columns=columnas_redundantes)

registrar(
    variable="DEPARTAMENTO_CONSULTADO / NIVEL_CONSULTADO",
    problema="100% redundantes: eran los parámetros de búsqueda usados en el scraping, ya representados por DEPARTAMENTO y NIVEL",
    transformacion="Eliminar ambas columnas",
    registros_afectados=len(df),
    justificacion="No aportan información adicional a DEPARTAMENTO y NIVEL; se elimina para reducir redundancia, asumiendo la pérdida menor de trazabilidad explícita del parámetro original de búsqueda."
)


Coincidencia DEPARTAMENTO_CONSULTADO vs DEPARTAMENTO: 100.0%
Coincidencia NIVEL_CONSULTADO vs NIVEL: 100.0%


#### Verificación



In [ ]:
columnas_limpiadas_olivier = ["CODIGO", "DEPARTAMENTO"]
for col in columnas_limpiadas_olivier:
    print(f"{col}: dtype={df[col].dtype}, nulos={df[col].isna().sum()} ({df[col].isna().mean()*100:.1f}%), unicos={df[col].nunique()}")

print("\n¿Columnas redundantes eliminadas?",
      all(c not in df.columns for c in ["DEPARTAMENTO_CONSULTADO", "NIVEL_CONSULTADO"]))


CODIGO: dtype=str, nulos=0 (0.0%), unicos=27413
DEPARTAMENTO: dtype=category, nulos=0 (0.0%), unicos=23

¿Columnas redundantes eliminadas? True


# Registro

In [ ]:
tabla_transformaciones = pd.DataFrame(registro_transformaciones)
tabla_transformaciones

,Problema detectado,Variable,Transformación,Registros afectados,Justificación
0,Ninguno,SECTOR,strip() + upper() + category,0,Dominio ya validado; solo se estandariza formato.
1,Ninguno,MODALIDAD,strip() + upper() + category,0,Dominio ya validado; solo se estandariza formato.
2,'SIN ESPECIFICAR' fuera del dominio inicial,AREA,Se agrega como categoría válida,5,"Es una categoría real del MINEDUC, no un vacío."
3,'TEMPORAL TITULOS' fuera del dominio inicial,STATUS,Se agrega como categoría válida,110,"Autorización real del MINEDUC, no error de cap..."
4,'INTERMEDIA' fuera del dominio inicial,JORNADA,Se agrega como categoría válida,205,"Jornada real del MINEDUC, no un error de captura."
5,3 categorías fuera del dominio inicial,PLAN,Se amplía el catálogo a 13 categorías reales,357,"Son modalidades reales del MINEDUC, no errores..."
6,Ninguno; un mismo DEPARTAMENTO puede tener var...,DEPARTAMENTAL,strip() + upper() + category,0,Representa la oficina administrativa del MINED...
7,Espacios/mayusculas inconsistentes y siglas de...,ESTABLECIMIENTO,strip() + upper() + colapsar espacios + envolv...,10,Estandariza formato sin alterar ortografia de ...
8,Posibles duplicados parciales (mismo plantel l...,ESTABLECIMIENTO,Deteccion por similitud de cadena con RapidFuz...,2766,No hay certeza suficiente para fusionar automa...
9,"Placeholders de dato faltante ('-', '.', etc.)...",DIRECCION,strip() + upper() + colapsar espacios; placeho...,36,Los placeholders no aportan informacion real; ...


In [ ]:
# Verificación rápida: tipo de dato y ausencia de nulos inesperados tras la limpieza
columnas_limpiadas = ["SECTOR", "AREA", "STATUS", "MODALIDAD", "JORNADA", "PLAN", "DEPARTAMENTAL"]
for col in columnas_limpiadas:
    print(f"{col}: dtype={df[col].dtype}, nulos={df[col].isna().sum()}, categorías={df[col].cat.categories.tolist()}")

SECTOR: dtype=category, nulos=0, categorías=['OFICIAL', 'PRIVADO', 'MUNICIPAL', 'COOPERATIVA']
AREA: dtype=category, nulos=0, categorías=['URBANA', 'RURAL', 'SIN ESPECIFICAR']
STATUS: dtype=category, nulos=0, categorías=['ABIERTA', 'CERRADA DEFINITIVAMENTE', 'CERRADA TEMPORALMENTE', 'TEMPORAL NOMBRAMIENTO', 'TEMPORAL TITULOS']
MODALIDAD: dtype=category, nulos=0, categorías=['MONOLINGUE', 'BILINGUE']
JORNADA: dtype=category, nulos=0, categorías=['MATUTINA', 'VESPERTINA', 'NOCTURNA', 'DOBLE', 'SIN JORNADA', 'INTERMEDIA']
PLAN: dtype=category, nulos=0, categorías=['DIARIO(REGULAR)', 'SABATINO', 'DOMINICAL', 'FIN DE SEMANA', 'IRREGULAR', 'MIXTO', 'A DISTANCIA', 'VIRTUAL A DISTANCIA', 'SEMIPRESENCIAL (FIN DE SEMANA)', 'SEMIPRESENCIAL (UN DÍA A LA SEMANA)', 'SEMIPRESENCIAL', 'SEMIPRESENCIAL (DOS DÍAS A LA SEMANA)', 'INTERCALADO']
DEPARTAMENTAL: dtype=category, nulos=0, categorías=['ALTA VERAPAZ', 'BAJA VERAPAZ', 'CHIMALTENANGO', 'CHIQUIMULA', 'EL PROGRESO', 'ESCUINTLA', 'GUATEMALA NORTE', 'G